# Diagnosis-Aware Report Generation

The classifier's predicted findings are written into the decoder's input before the report
starts, so the decoder is told what to describe instead of having to infer it. This targets
recall, which the previous run measured at 0.120.

Prerequisite: run `python predict_findings.py` once, which writes `prompt_probs.csv`
and `prompt_thresholds.json`.

Set `USE_ORACLE = True` to prompt with the true labels instead of the predicted ones.
That run is not a deployable system - it measures the ceiling this approach could reach
if the classifier were perfect, which is a useful third row in the ablation table.

## 0. Create the helper module

In [ ]:
%%writefile rgp_data.py
import json
import os
import re
from collections import Counter

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

IMG_SIZE = 224
MAX_LEN = 100
PROMPT_LEN = 16
MIN_FREQ = 3

PAD, BOS, EOS, UNK, SEP = 0, 1, 2, 3, 4
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>", "<sep>"]

LABEL_COLS = [
    "No Finding",
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion",
    "Lung Opacity", "Pleural Effusion", "Pleural Other",
    "Pneumonia", "Pneumothorax",
]

_NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    _NORM,
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    _NORM,
])


def tokenize(text):
    t = str(text).lower()
    t = re.sub(r"[^a-z0-9%./\- ]", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip().split()


class Vocab:
    def __init__(self, itos):
        self.itos = list(itos)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens, max_len=MAX_LEN):
        ids = [self.stoi.get(t, UNK) for t in tokens][: max_len - 1] + [EOS]
        return ids + [PAD] * (max_len - len(ids))

    def decode(self, ids):
        words = []
        for i in ids:
            i = int(i)
            if i == EOS:
                break
            if i > SEP:
                words.append(self.itos[i])
        return words

    def save(self, path):
        json.dump(self.itos, open(path, "w"))

    @staticmethod
    def load(path):
        return Vocab(json.load(open(path)))

    @staticmethod
    def build(texts, min_freq=MIN_FREQ):
        c = Counter()
        for t in texts:
            c.update(tokenize(t))
        for lab in LABEL_COLS:
            c.update(tokenize(lab))
        words = sorted([w for w, n in c.items() if n >= min_freq])
        return Vocab(SPECIALS + words), c


def make_prompt_ids(vocab, findings, prompt_len=PROMPT_LEN):
    ids = [BOS]
    for f in findings:
        ids += [vocab.stoi.get(w, UNK) for w in tokenize(f)]
    ids = ids[: prompt_len - 1]
    ids.append(SEP)
    return ids + [PAD] * (prompt_len - len(ids))


class PromptedReportDataset(Dataset):
    def __init__(self, df, vocab, tf, max_len=MAX_LEN, prompt_len=PROMPT_LEN):
        self.paths = df["local_path"].tolist()
        self.reports = df["report"].tolist()
        self.prompts = df["prompt_findings"].tolist()
        self.vocab = vocab
        self.tf = tf
        self.max_len = max_len
        self.prompt_len = prompt_len

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        for k in range(10):
            j = (i + k) % len(self.paths)
            try:
                img = Image.open(self.paths[j]).convert("RGB")
            except Exception:
                continue
            findings = [f for f in str(self.prompts[j]).split("|") if f]
            pid = make_prompt_ids(self.vocab, findings, self.prompt_len)
            rid = self.vocab.encode(tokenize(self.reports[j]), self.max_len)
            return (self.tf(img),
                    torch.tensor(pid, dtype=torch.long),
                    torch.tensor(rid, dtype=torch.long))
        raise RuntimeError("too many unreadable images near index %d" % i)


def attach_prompts(df, probs, thresholds, use_oracle=False, max_findings=3):
    if use_oracle:
        def row_findings(r):
            pos = [c for c in LABEL_COLS[1:] if r.get(c, 0) == 1]
            return "|".join(pos) if pos else "No Finding"
        df = df.copy()
        df["prompt_findings"] = df.apply(row_findings, axis=1)
        return df

    df = df.merge(probs, on="dicom_id", how="left")
    pcols = [f"p_{c}" for c in LABEL_COLS]
    for c in pcols:
        df[c] = df[c].fillna(0.0)

    def row_findings(r):
        hits = [(c, r[f"p_{c}"]) for c in LABEL_COLS[1:]
                if r[f"p_{c}"] >= thresholds.get(c, 0.5)]
        hits.sort(key=lambda x: -x[1])
        names = [c for c, _ in hits[:max_findings]]
        return "|".join(names) if names else "No Finding"

    df["prompt_findings"] = df.apply(row_findings, axis=1)
    return df


def load_pairs(data_dir=".", use_oracle=False, one_image_per_study=True,
               val_fraction=0.05, seed=42):
    pairs = pd.read_csv(os.path.join(data_dir, "image_report_pairs.csv"))
    avail = pd.read_csv(os.path.join(data_dir, "available_labels.csv"))

    df = pairs.merge(avail[["dicom_id", "local_path"]], on="dicom_id", how="inner")
    df = df[df["report"].notna()]
    if one_image_per_study:
        df = df.sort_values("dicom_id").drop_duplicates(subset=["study_id"])
    df = df.reset_index(drop=True)

    probs = pd.read_csv(os.path.join(data_dir, "prompt_probs.csv"))
    thresholds = json.load(open(os.path.join(data_dir, "prompt_thresholds.json")))
    df = attach_prompts(df, probs, thresholds, use_oracle=use_oracle)

    train = df[df["split"] == "train"].reset_index(drop=True)
    val = df[df["split"] == "validate"].reset_index(drop=True)
    test = df[df["split"] == "test"].reset_index(drop=True)

    if len(val) < 500:
        extra = train.sample(n=int(len(train) * val_fraction), random_state=seed)
        train = train.drop(extra.index).reset_index(drop=True)
        val = pd.concat([val, extra], ignore_index=True)

    return train, val, test

In [ ]:
%%writefile rg_metrics.py
import math
from collections import Counter


def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def corpus_bleu(hyps, refs, max_n=4):
    num = [0] * max_n
    den = [0] * max_n
    hyp_len = 0
    ref_len = 0

    for h, r in zip(hyps, refs):
        hyp_len += len(h)
        ref_len += len(r)
        for n in range(1, max_n + 1):
            hn = _ngrams(h, n)
            rn = _ngrams(r, n)
            num[n - 1] += sum(min(c, rn[g]) for g, c in hn.items())
            den[n - 1] += max(sum(hn.values()), 0)

    scores = []
    log_sum = 0.0
    for n in range(max_n):
        p = (num[n] + 1e-9) / (den[n] + 1e-9)
        log_sum += math.log(p)
        bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
        scores.append(bp * math.exp(log_sum / (n + 1)))
    return scores


def _lcs(a, b):
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        cur = [0] * (n + 1)
        ai = a[i - 1]
        for j in range(1, n + 1):
            cur[j] = prev[j - 1] + 1 if ai == b[j - 1] else max(prev[j], cur[j - 1])
        prev = cur
    return prev[n]


def rouge_l(hyps, refs, beta=1.2):
    total = 0.0
    for h, r in zip(hyps, refs):
        if not h or not r:
            continue
        l = _lcs(h, r)
        if l == 0:
            continue
        p = l / len(h)
        rc = l / len(r)
        total += ((1 + beta ** 2) * p * rc) / (rc + beta ** 2 * p)
    return total / max(len(hyps), 1)


def evaluate_text(hyps, refs):
    b = corpus_bleu(hyps, refs)
    return {
        "BLEU-1": b[0], "BLEU-2": b[1], "BLEU-3": b[2], "BLEU-4": b[3],
        "ROUGE-L": rouge_l(hyps, refs),
    }

## 1. Setup

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import densenet121

from rgp_data import (Vocab, PromptedReportDataset, load_pairs, tokenize, make_prompt_ids,
                      train_tf, eval_tf, LABEL_COLS, PAD, BOS, EOS, UNK, SEP,
                      MAX_LEN, PROMPT_LEN)
from rg_metrics import evaluate_text

DATA_DIR     = "."
USE_ORACLE   = True
CKPT_DIR     = "checkpoints_prompt_oracle" if USE_ORACLE else "checkpoints_prompt"
CLS_CKPT     = "checkpoints/best.pt"

BATCH_SIZE   = 16
EPOCHS       = 8
LR           = 1e-4
ENCODER_LR   = 1e-5
D_MODEL      = 512
N_HEAD       = 8
N_LAYERS     = 3
FF_DIM       = 2048
DROPOUT      = 0.1
NUM_WORKERS  = 2
EVAL_SUBSET  = 500
BEAM         = 3
LEN_PENALTY  = 1.0
SEED         = 42

os.makedirs(CKPT_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| oracle prompts:", USE_ORACLE)

## 2. Data and prompts

In [ ]:
train_df, val_df, test_df = load_pairs(DATA_DIR, use_oracle=USE_ORACLE)
print(f"train {len(train_df)}   val {len(val_df)}   test {len(test_df)}")

n_find = train_df["prompt_findings"].apply(lambda s: 0 if s == "No Finding" else len(s.split("|")))
print(f"\nFindings per prompt: mean {n_find.mean():.2f}  median {n_find.median():.0f}  max {n_find.max()}")
print(f"Prompts that are empty (No Finding): {(n_find == 0).mean() * 100:.1f}%")

flat = train_df["prompt_findings"].str.split("|").explode()
print("\nHow often each finding is prompted:")
print(flat[flat != "No Finding"].value_counts().to_string())

## 3. Vocabulary

In [ ]:
vocab_path = os.path.join(CKPT_DIR, "vocab.json")
if os.path.isfile(vocab_path):
    vocab = Vocab.load(vocab_path)
    print("Loaded existing vocabulary")
else:
    vocab, _ = Vocab.build(train_df["report"].tolist())
    vocab.save(vocab_path)
print("Vocabulary size:", len(vocab))

demo = make_prompt_ids(vocab, ["Cardiomegaly", "Pleural Effusion"])
print("\nExample prompt ids :", demo)
print("Example prompt back:", vocab.decode(demo))

In [ ]:
train_loader = DataLoader(PromptedReportDataset(train_df, vocab, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=True, persistent_workers=NUM_WORKERS > 0)
val_loader   = DataLoader(PromptedReportDataset(val_df, vocab, eval_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=NUM_WORKERS > 0)

xb, pb, rb = next(iter(train_loader))
print("images:", tuple(xb.shape), " prompt:", tuple(pb.shape), " report:", tuple(rb.shape))

## 4. Model

The decoder input is the prompt block followed by the report. Loss is computed only on the
report positions, so the model is never trained to reproduce the prompt - the prompt is
context, not a target. At generation time the decoder is seeded with the prompt block and
continues from there.

In [ ]:
TOTAL_LEN = PROMPT_LEN + MAX_LEN


class PromptedGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, nhead=N_HEAD, layers=N_LAYERS,
                 ff=FF_DIM, dropout=DROPOUT, max_len=TOTAL_LEN):
        super().__init__()
        backbone = densenet121(weights=None)
        backbone.classifier = nn.Linear(backbone.classifier.in_features, 13)
        if os.path.isfile(CLS_CKPT):
            ck = torch.load(CLS_CKPT, map_location="cpu", weights_only=False)
            backbone.load_state_dict(ck["model"])
            print(f"Encoder initialised from {CLS_CKPT} (AUC {ck.get('auc', float('nan')):.4f})")

        self.features = backbone.features
        self.proj = nn.Linear(1024, d_model)
        self.img_pos = nn.Parameter(torch.zeros(1, 49, d_model))

        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.txt_pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        self.drop = nn.Dropout(dropout)

        layer = nn.TransformerDecoderLayer(d_model, nhead, ff, dropout,
                                           batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(layer, layers)
        self.out = nn.Linear(d_model, vocab_size)

        nn.init.normal_(self.img_pos, std=0.02)
        nn.init.normal_(self.txt_pos, std=0.02)

    def encode(self, img):
        f = self.features(img)
        f = torch.relu(f).flatten(2).transpose(1, 2)
        return self.proj(f) + self.img_pos

    def decode(self, memory, tgt_in):
        t = tgt_in.size(1)
        x = self.drop(self.emb(tgt_in) + self.txt_pos[:, :t])
        mask = torch.triu(torch.ones(t, t, dtype=torch.bool, device=tgt_in.device), 1)
        h = self.decoder(x, memory, tgt_mask=mask)
        return self.out(h)

    def forward(self, img, tgt_in):
        return self.decode(self.encode(img), tgt_in)

    @torch.no_grad()
    def generate(self, img, prefix, max_new=MAX_LEN, beam=1, length_penalty=1.0):
        dev = img.device
        memory = self.encode(img)
        b = img.size(0)

        if beam <= 1:
            ids = prefix.clone()
            done = torch.zeros(b, dtype=torch.bool, device=dev)
            for _ in range(max_new):
                nxt = self.decode(memory, ids)[:, -1].argmax(-1)
                nxt[done] = PAD
                ids = torch.cat([ids, nxt.unsqueeze(1)], dim=1)
                done = done | (nxt == EOS)
                if done.all():
                    break
            return ids[:, prefix.size(1):]

        n, d = memory.size(1), memory.size(2)
        V = self.out.out_features
        memory = memory.unsqueeze(1).expand(b, beam, n, d).reshape(b * beam, n, d)

        P = prefix.size(1)
        ids = prefix.unsqueeze(1).expand(b, beam, P).reshape(b * beam, P).clone()
        scores = torch.full((b, beam), -1e9, device=dev)
        scores[:, 0] = 0.0
        scores = scores.view(-1)
        fin = torch.zeros(b * beam, dtype=torch.bool, device=dev)
        ar = torch.arange(b, device=dev).unsqueeze(1)

        for _ in range(max_new):
            lp = torch.log_softmax(self.decode(memory, ids)[:, -1].float(), -1)
            lp[fin] = -1e9
            lp[fin, PAD] = 0.0
            cand = (scores.unsqueeze(1) + lp).view(b, beam * V)
            top, idx = cand.topk(beam, dim=-1)
            bi = torch.div(idx, V, rounding_mode="floor")
            tok = idx % V
            g = (ar * beam + bi).view(-1)
            ids = torch.cat([ids[g], tok.view(-1, 1)], dim=1)
            scores = top.reshape(-1)
            fin = fin[g] | (tok.view(-1) == EOS)
            if fin.all():
                break

        gen = ids[:, P:]
        lens = (gen != PAD).sum(1).clamp(min=1).float()
        best = (scores / lens.pow(length_penalty)).view(b, beam).argmax(-1)
        sel = torch.arange(b, device=dev) * beam + best
        return gen[sel]

In [ ]:
model = PromptedGenerator(len(vocab)).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW([
    {"params": model.features.parameters(), "lr": ENCODER_LR},
    {"params": [p for n, p in model.named_parameters() if not n.startswith("features.")], "lr": LR},
], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

## 5. Train

In [ ]:
def step_loss(model, img, prompt, report):
    full = torch.cat([prompt, report], dim=1)
    tgt_in = full[:, :-1]
    tgt_out = full[:, 1:].clone()
    tgt_out[:, : prompt.size(1) - 1] = PAD
    logits = model(img, tgt_in)
    return criterion(logits.reshape(-1, logits.size(-1)).float(), tgt_out.reshape(-1))


@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    for img, prompt, report in loader:
        img, prompt, report = (img.to(device, non_blocking=True),
                               prompt.to(device, non_blocking=True),
                               report.to(device, non_blocking=True))
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            loss = step_loss(model, img, prompt, report)
        total += loss.item() * img.size(0); n += img.size(0)
    return total / max(n, 1)


@torch.no_grad()
def eval_text(model, df, limit=EVAL_SUBSET):
    model.eval()
    sub = df.iloc[:limit]
    loader = DataLoader(PromptedReportDataset(sub, vocab, eval_tf), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, pin_memory=True)
    hyps, refs = [], []
    for img, prompt, report in loader:
        img, prompt = img.to(device, non_blocking=True), prompt.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            gen = model.generate(img, prompt, beam=BEAM, length_penalty=LEN_PENALTY)
        for g, r in zip(gen.cpu(), report):
            hyps.append(vocab.decode(g))
            refs.append(vocab.decode(r))
    return evaluate_text(hyps, refs), hyps, refs

In [ ]:
history, best_b4, start_ep = [], 0.0, 1
last_path = os.path.join(CKPT_DIR, "last.pt")

if os.path.isfile(last_path):
    ck = torch.load(last_path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    start_ep, best_b4, history = ck["epoch"] + 1, ck["best_b4"], ck.get("history", [])
    print(f"Resumed from epoch {ck['epoch']}")

In [ ]:
for epoch in range(start_ep, EPOCHS + 1):
    model.train()
    running, seen, t0 = 0.0, 0, time.time()

    for step, (img, prompt, report) in enumerate(train_loader, 1):
        img, prompt, report = (img.to(device, non_blocking=True),
                               prompt.to(device, non_blocking=True),
                               report.to(device, non_blocking=True))
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            loss = step_loss(model, img, prompt, report)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()

        running += loss.item() * img.size(0); seen += img.size(0)
        if step % 200 == 0:
            print(f"  ep{epoch} {step}/{len(train_loader)}  loss {running/seen:.4f}"
                  f"  {seen/(time.time()-t0):.1f} img/s")

    tr_loss = running / seen
    vl_loss = eval_loss(model, val_loader)
    scores, _, _ = eval_text(model, val_df)
    scheduler.step(scores["BLEU-4"])

    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": vl_loss, "scores": scores})
    print(f"epoch {epoch}: train {tr_loss:.4f}  val {vl_loss:.4f}  "
          + "  ".join(f"{k} {v:.4f}" for k, v in scores.items())
          + f"  ({(time.time()-t0)/60:.1f} min)")

    tmp = last_path + ".tmp"
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "best_b4": max(best_b4, scores["BLEU-4"]),
                "history": history}, tmp)
    os.replace(tmp, last_path)

    if scores["BLEU-4"] > best_b4:
        best_b4 = scores["BLEU-4"]
        bp = os.path.join(CKPT_DIR, "best.pt")
        torch.save({"model": model.state_dict(), "scores": scores}, bp + ".tmp")
        os.replace(bp + ".tmp", bp)
        print(f"    new best saved (BLEU-4 {best_b4:.4f})")

json.dump(history, open(os.path.join(CKPT_DIR, "history.json"), "w"), indent=2)

## 6. Test set

In [ ]:
ck = torch.load(os.path.join(CKPT_DIR, "best.pt"), map_location=device, weights_only=False)
model.load_state_dict(ck["model"])

eval_df = test_df if len(test_df) >= 200 else val_df
scores, hyps, refs = eval_text(model, eval_df, limit=2000)

print(f"n = {min(len(eval_df), 2000)}\n")
print(pd.DataFrame([
    {"model": "prompted (this run)", **{k: round(v, 3) for k, v in scores.items()}},
    {"model": "no prompting (run 3)", "BLEU-1": 0.213, "BLEU-2": 0.120, "BLEU-3": 0.075,
     "BLEU-4": 0.050, "ROUGE-L": 0.178},
    {"model": "R2Gen (MIMIC)", "BLEU-1": 0.353, "BLEU-2": 0.218, "BLEU-3": 0.145,
     "BLEU-4": 0.103, "ROUGE-L": 0.277},
]).to_string(index=False))

uniq = len({" ".join(h) for h in hyps})
print(f"\nDistinct generated reports: {uniq} / {len(hyps)} ({uniq/len(hyps)*100:.1f}%)")

## 7. Clinical efficacy — the number that matters

In [ ]:
from clinical_eval import clinical_efficacy, LABELS

hyp_texts = [" ".join(h) for h in hyps]
true = eval_df.iloc[:len(hyp_texts)][LABELS].fillna(0).replace(-1, 0).values

per_class, summary = clinical_efficacy(hyp_texts, true)
print("THIS RUN :", summary)
print("RUN 3    : {'macro_f1': 0.1161, 'micro_precision': 0.4802, "
      "'micro_recall': 0.1196, 'micro_f1': 0.1915}")
print()
print(per_class.sort_values("support", ascending=False).to_string(index=False))

In [ ]:
prev = {"Pleural Effusion": 0.308, "Cardiomegaly": 0.060, "Lung Opacity": 0.057,
        "Atelectasis": 0.123, "Edema": 0.160, "Pneumonia": 0.061, "Consolidation": 0.120,
        "Enlarged Cardiomediastinum": 0.0, "Lung Lesion": 0.0, "Pneumothorax": 0.045,
        "Fracture": 0.0, "Pleural Other": 0.0}

cmp = per_class.set_index("finding")
rows = [{"finding": k, "recall_run3": v,
         "recall_prompted": round(float(cmp.loc[k, "recall"]), 3),
         "change": round(float(cmp.loc[k, "recall"]) - v, 3)}
        for k, v in prev.items() if k in cmp.index]
print(pd.DataFrame(rows).sort_values("change", ascending=False).to_string(index=False))

## 8. Sample outputs

In [ ]:
import random
random.seed(0)
for i in random.sample(range(len(hyps)), 5):
    print("PROMPT:", eval_df.iloc[i]["prompt_findings"])
    print("REF   :", " ".join(refs[i]))
    print("GEN   :", " ".join(hyps[i]))
    print("-" * 90)